# Лабораторная работа №8
## Прогнозирование временного ряда (ARIMA, символьная регрессия, МГУА)

**Задание:** временной ряд → визуализация и характеристики → разбиение train/test → прогнозы: **ARIMA**, **символьная регрессия**, **GMDH (Combi + Mia)** → графики и метрики.

> Установите зависимости: `pip install statsmodels gplearn` . Пакет `gmdh` — как в ЛР7 (может потребоваться Boost / сборка C++).


In [ ]:
# pip install statsmodels gplearn
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)


## 1. Данные: годовой ряд числа солнечных пятен (Sunspots), statsmodels

Классический одномерный ряд для демонстрации AR/ARIMA.


In [ ]:
import statsmodels.api as sm

sun = sm.datasets.sunspots.load_pandas().data.dropna()
y = sun["SUNACTIVITY"].astype(float).to_numpy()
t_idx = sun["YEAR"].astype(float).to_numpy()

series = pd.Series(y, index=t_idx, name="sunspots")
print(series.describe())
display(series.to_frame().head())


## 2. Визуализация ряда и основные характеристики


In [ ]:
from sklearn.metrics import mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

series.plot(ax=axes[0, 0], color="steelblue", lw=1.2)
axes[0, 0].set_title("Временной ряд (Sunspots)")
axes[0, 0].set_xlabel("Год")
axes[0, 0].set_ylabel("Активность")

axes[0, 1].hist(series.values, bins=30, color="coral", edgecolor="white")
axes[0, 1].set_title("Гистограмма значений")

plot_acf(series, lags=40, ax=axes[1, 0])
plot_pacf(series, lags=40, method="ywm", ax=axes[1, 1])

plt.tight_layout()
plt.show()


## 3. Разбиение на обучение и тест (хронологическое)

Последние 20% — тест.


In [ ]:
TEST_FRAC = 0.2
split = int(len(y) * (1.0 - TEST_FRAC))

y_train_raw = y[:split]
y_test_raw = y[split:]
t_train = t_idx[:split]
t_test = t_idx[split:]

print(f"Train: n={len(y_train_raw)}, года {t_train[0]:.0f}—{t_train[-1]:.0f}")
print(f"Test:  n={len(y_test_raw)}, года {t_test[0]:.0f}—{t_test[-1]:.0f}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_train, y_train_raw, label="Train", color="C0")
ax.plot(t_test, y_test_raw, label="Test", color="C1")
ax.axvline(t_test[0], color="gray", ls="--", lw=1, label="Split")
ax.legend()
ax.set_title("Разбиение временного ряда")
ax.set_xlabel("Год")
plt.tight_layout()
plt.show()


## 4. ARIMA

Подбор $(p,0,q)$ с константой по минимальному AIC (небольшой перебор).


In [ ]:
from statsmodels.tsa.arima.model import ARIMA

best_aic = np.inf
best_order = (2, 0, 2)
for p in range(0, 4):
    for q in range(0, 4):
        try:
            m = ARIMA(y_train_raw, order=(p, 0, q), trend="c")
            res = m.fit(method_kwargs={"warn_convergence": False})
            if res.aic < best_aic:
                best_aic, best_order = res.aic, (p, 0, q)
        except Exception:
            pass

arima_res = ARIMA(y_train_raw, order=best_order, trend="c").fit()
pred_arima = arima_res.forecast(steps=len(y_test_raw))
mae_arima = mean_absolute_error(y_test_raw, pred_arima)

print("ARIMA order:", best_order, "AIC=", round(arima_res.aic, 2))
print("MAE тест (ARIMA):", round(mae_arima, 4))


## 5. Лаговая модель для ML: символьная регрессия и МГУА

Строка наблюдения на момент $t$: $X=(y_{t-L},\ldots,y_{t-1})$, цель $y_t$. Обучение только на train-интервале; тест — **one-step ahead** с лагами из **истинного** прошлого ряда.


In [ ]:
LAG = 15
split_time = t_test[0]

lags_X, lags_y, lags_time = [], [], []
for t in range(LAG, len(y)):
    lags_X.append(y[t - LAG : t])
    lags_y.append(y[t])
    lags_time.append(t_idx[t])

X_all = np.asarray(lags_X, float)
y_all = np.asarray(lags_y, float)
tl = np.asarray(lags_time)

tr_mask = tl < split_time
te_mask = tl >= split_time

X_tr, y_tr = X_all[tr_mask], y_all[tr_mask]
X_te, y_te_ml = X_all[te_mask], y_all[te_mask]
t_te = tl[te_mask]

print("X_train", X_tr.shape, "X_test", X_te.shape)


## 6. Символьная регрессия (`gplearn`)


In [ ]:
from gplearn.genetic import SymbolicRegressor

sym = SymbolicRegressor(
    population_size=800,
    generations=15,
    stopping_criteria=0.01,
    parsimonious_coefficient=0.001,
    random_state=42,
    n_jobs=-1,
)

sym.fit(X_tr, y_tr)
pred_sym = sym.predict(X_te)
mae_sym = mean_absolute_error(y_te_ml, pred_sym)
print("MAE символьная регрессия:", round(mae_sym, 4))
print(sym._program)


## 7. МГУА: Combi + Mia (`gmdh`)


In [ ]:
GMDH_OK = False
mae_combi = mae_mia = np.nan

try:
    import gmdh
    from gmdh import Combi, Mia, Criterion, CriterionType

    crit = gmdh.Criterion(criterion_type=CriterionType.REGULARITY)

    combi = Combi()
    combi.fit(
        X_tr.astype(np.float64),
        y_tr.astype(np.float64),
        criterion=crit,
        test_size=0.25,
        n_jobs=-1,
        verbose=0,
    )
    pred_combi = combi.predict(X_te.astype(np.float64))
    mae_combi = mean_absolute_error(y_te_ml, pred_combi)

    mia = Mia()
    mia.fit(
        X_tr.astype(np.float64),
        y_tr.astype(np.float64),
        criterion=crit,
        k_best=3,
        polynomial_type=gmdh.PolynomialType.QUADRATIC,
        test_size=0.25,
        n_jobs=-1,
        verbose=0,
    )
    pred_mia = mia.predict(X_te.astype(np.float64))
    mae_mia = mean_absolute_error(y_te_ml, pred_mia)
    GMDH_OK = True
    print("MAE Combi:", round(mae_combi, 4))
    print("MAE Mia:", round(mae_mia, 4))
except Exception as e:
    pred_combi = pred_mia = None
    print("GMDH недоступен:", repr(e))


## 8. Сводная таблица: MAE на тесте


In [ ]:
rows = [
    {"Метод": "ARIMA", "MAE": mean_absolute_error(y_test_raw, pred_arima)},
    {"Метод": "Символьная регрессия", "MAE": mae_sym},
    {"Метод": "GMDH Combi", "MAE": mae_combi},
    {"Метод": "GMDH Mia", "MAE": mae_mia},
]
display(pd.DataFrame(rows).sort_values("MAE"))


## 9. График: тест и прогнозы


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# y_te_ml совпадает с y_test_raw при одинаковом разбиении; ось времени — t_test
ax.plot(t_test, y_test_raw, color="black", lw=2, label="Факт (тест)")

ax.plot(t_test, pred_arima, ls="--", color="C0", label="ARIMA")
ax.plot(t_test, pred_sym, ls="--", color="C1", alpha=0.95, label="Символьная регрессия")

if GMDH_OK:
    ax.plot(t_test, pred_combi, alpha=0.9, color="C2", label="GMDH Combi")
    ax.plot(t_test, pred_mia, alpha=0.9, color="C3", label="GMDH Mia")

ax.set_title("Тестовый интервал: факт и прогнозы")
ax.set_xlabel("Год")
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()


## 10. Выводы и обратная связь по `gmdh`

- Ряд Sunspots: график, распределение, ACF/PACF; split 80/20 по времени.
- Прогнозы: ARIMA; символьная регрессия; **Combi** (линейный) и **Mia** (нелинейный) из `gmdh`.
- Метрика: **MAE**.

**Telegram / ТМО_МГУА:** баги установки (скрин), опечатки (*specitying* в `CriterionType`), трудности.
